In [1]:
# Calculate and save MDA8

In [1]:
import os
import xarray as xr
import numpy as np
import json
from utils.utils import get_scenario_config

In [2]:
def load_file_list(DIR, filename):
    file_path = os.path.join(DIR, filename)
    with open(file_path, "r") as f:
        data = json.load(f)
    return data["files"]

In [3]:
# === Processing function ===
def calculate_monthly_mean_8hrdailymax(start_date, end_date, o3_surf):
    daterange = xr.date_range(start_date, end_date, calendar="noleap", use_cftime=True)

    MDA8 = xr.DataArray(  # Maximum Daily 8hr Average O3
        np.nan,
        dims=["time", "lat", "lon"],
        coords={"time": daterange, "lat": o3_surf.lat, "lon": o3_surf.lon},
    )

    for i in range(len(daterange)):
        date = daterange[i].strftime("%Y-%m-%d")
        o3_day = o3_surf.sel(time=slice(date + " 00:00:00", date + " 23:00:00"))
        o3_rolling = o3_day.rolling(time=8).mean()
        MDA8[i, :, :] = o3_rolling.max("time")

    monthly_mean = MDA8.resample(time="ME").mean()
    return monthly_mean

In [4]:
# === Path config ===
FILE_DIR = "/glade/work/awells/air_quality/CESM/ozone/file_paths/"
SAVE_DIR = "/glade/work/awells/air_quality/CESM/ozone/MDA8/"

# Set to whatever scenario you want, function returns error if not recognised
scenario = "SSP245_G6"

config = get_scenario_config(scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

# === Main loop ===
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")
    file_list = load_file_list(FILE_DIR, f"file_list_{scenario}_{ens_num}.json")
    monthly_means = []

    for f in file_list:
        if not os.path.exists(f):
            raise ValueError(f"Missing: {f}")

        print(f"Reading {os.path.basename(f)}")
        ds = xr.open_dataset(f)["O3_SRF"]

        start_date = str(ds.time[0].values)[:10]  # e.g. yyyy-mm-dd
        end_date = str(ds.time[-1].values)[:10]
        end_time = str(ds.time[-1].values)[11:16]  # e.g. hh:mm
        midnight = "00:00"

        if end_time == midnight:
            print("changing final time step")
            # making final time step 23:00
            end_date = str(ds.time[-2].values)[:10]

        mm = calculate_monthly_mean_8hrdailymax(start_date, end_date, ds)

        # Trim to start_year-01-01 - end_year-12-31
        mm = mm.sel(time=slice(f"{years.start}-01-01", f"{years.stop}-12-31"))

        monthly_means.append(mm)

    if monthly_means:
        combined = xr.concat(monthly_means, dim="time")

        dates = f"{years.start}0101-{years.stop}1231"

        out_file = f"MDA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving to {out_path}")
        description = ("MDA8: 8hr Daily Maximum Calculates the 8-hour "
                       "daily maximum surface ozone concentration - scripts "
                       "by A.F. Wells (2025)")
        combined.attrs = ds.attrs
        combined.attrs["description"] = description
        combined.attrs["ensemble_number"] = ens_num
        combined.attrs["scenario"] = scenario
        combined.to_netcdf(out_path)

print("All processing complete.")


Processing SSP245_G6, Ensemble 01
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.001.cam.h4.O3_SRF.2015010100-2025010100.nc
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.001.cam.h4.O3_SRF.2025010100-2035010100.nc
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.001.cam.h4.O3_SRF.2035010100-2045010100.nc
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.001.cam.h4.O3_SRF.2045010100-2055010100.nc
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.001.cam.h4.O3_SRF.2055010100-2065010100.nc
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.001.cam.h4.O3_SRF.2065010100-2075010100.nc
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.001.cam.h4.O3_SRF.2075010100-2085010100.nc
Saving to /glade/work/awells/air_quality/CESM/ozone/MDA8/MDA8_CESM2_SSP245_G6_01_20200101-20841231.nc
Processing SSP245_G6, Ensemble 02
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.002.cam.h4.O3_SRF.2015010100-2025010100.nc
Reading b.e21.BWSSP245cmip6.f0